In [ ]:
import argparse
import re
import json
import warnings
import optuna
import sage
import shap
import itertools
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

import xgboost as xgb
from xgboost import XGBClassifier

import sklearn
from sklearn.model_selection import cross_val_predict, cross_val_score, StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from sklearn.feature_selection import RFECV
from sklearn.dummy import DummyClassifier

# Accelerate sklearn operations on Intel CPUs
# from sklearnex import patch_sklearn
# patch_sklearn()

# Ignore warnings for cleaner output
warnings.simplefilter(action='ignore')

### data loading and preparation

In [ ]:
mft_list_5 = ['care', 'fairness','loyalty','authority','purity']
mft_list = ['care','harm','fairness','cheating','loyalty','betrayal','authority','subversion','purity','degradation']

In [ ]:
# Load 200 English songs dataset 10 values
df = pd.read_csv(f"datasets/m_dataset_en200.csv")
df = df.drop(["youtube_id"], axis=1)

In [ ]:
#@title create shorted and better codified names for audio features
better_feat_names = [
    'dynamic_complexity',
    'loudness_integrated',
    'melody.pitch_range',
    'melody.direction',
    'melody.pitch_height',
    'melody.step_size_mean',
    'melody.step_size_std',
    'melody.hist_interval_0',
    'melody.hist_interval_1',
    'melody.hist_interval_2',
    'melody.hist_interval_3',
    'melody.hist_interval_4',
    'melody.hist_interval_5',
    'melody.hist_interval_6',
    'melody.hist_interval_7',
    'melody.hist_interval_8',
    'melody.hist_interval_9',
    'melody.hist_interval_10',
    'melody.hist_interval_11',
    'beats_loudness.mean',
    'beats_loudness.std',
    'rhythm.tempo',
    'bpm.hist_firstPeakBPM',
    'bpm.hist_firstPeakWeight',
    'bpm.hist_secondPeakBPM',
    'bpm.hist_secondPeakWeight',
    'rhythm.danceability',
    'rhythm.onset_rate',
    'mfcc_mean_0',
    'mfcc_mean_1',
    'mfcc_mean_2',
    'mfcc_mean_3',
    'mfcc_mean_4',
    'mfcc_mean_5',
    'mfcc_mean_6',
    'mfcc_mean_7',
    'mfcc_mean_8',
    'mfcc_mean_9',
    'mfcc_mean_10',
    'mfcc_mean_11',
    'mfcc_std_0',
    'mfcc_std_1',
    'mfcc_std_2',
    'mfcc_std_3',
    'mfcc_std_4',
    'mfcc_std_5',
    'mfcc_std_6',
    'mfcc_std_7',
    'mfcc_std_8',
    'mfcc_std_9',
    'mfcc_std_10',
    'mfcc_std_11',
    'delta_mfcc_mean_0',
    'delta_mfcc_mean_1',
    'delta_mfcc_mean_2',
    'delta_mfcc_mean_3',
    'delta_mfcc_mean_4',
    'delta_mfcc_mean_5',
    'delta_mfcc_mean_6',
    'delta_mfcc_mean_7',
    'delta_mfcc_mean_8',
    'delta_mfcc_mean_9',
    'delta_mfcc_mean_10',
    'delta_mfcc_mean_11',
    'delta_mfcc_std_0',
    'delta_mfcc_std_1',
    'delta_mfcc_std_2',
    'delta_mfcc_std_3',
    'delta_mfcc_std_4',
    'delta_mfcc_std_5',
    'delta_mfcc_std_6',
    'delta_mfcc_std_7',
    'delta_mfcc_std_8',
    'delta_mfcc_std_9',
    'delta_mfcc_std_10',
    'delta_mfcc_std_11',
    'spectral_flatness.mean',
    'spectral_flatness.std',
    'spectral_kurtosis.mean',
    'spectral_kurtosis.std',
    'spectral_skewness.mean',
    'spectral_skewness.std',
    'spectral_spread.mean',
    'spectral_spread.std',
    'spectral_centroid.mean',
    'spectral_centroid.std',
    'spectral_complexity.mean',
    'spectral_complexity.std',
    'spectral_contrast_coeffs.mean_0',
    'spectral_contrast_coeffs.mean_1',
    'spectral_contrast_coeffs.mean_2',
    'spectral_contrast_coeffs.mean_3',
    'spectral_contrast_coeffs.mean_4',
    'spectral_contrast_coeffs.mean_5',
    'spectral_contrast_coeffs.std_0',
    'spectral_contrast_coeffs.std_1',
    'spectral_contrast_coeffs.std_2',
    'spectral_contrast_coeffs.std_3',
    'spectral_contrast_coeffs.std_4',
    'spectral_contrast_coeffs.std_5',
    'spectral_contrast_valleys.mean_0',
    'spectral_contrast_valleys.mean_1',
    'spectral_contrast_valleys.mean_2',
    'spectral_contrast_valleys.mean_3',
    'spectral_contrast_valleys.mean_4',
    'spectral_contrast_valleys.mean_5',
    'spectral_contrast_valleys.std_0',
    'spectral_contrast_valleys.std_1',
    'spectral_contrast_valleys.std_2',
    'spectral_contrast_valleys.std_3',
    'spectral_contrast_valleys.std_4',
    'spectral_contrast_valleys.std_5',
    'spectral_flux.mean',
    'spectral_flux.std',
    'zerocrossingrate.mean',
    'zerocrossingrate.std',
    'chromagram_mean_0',
    'chromagram_mean_1',
    'chromagram_mean_2',
    'chromagram_mean_3',
    'chromagram_mean_4',
    'chromagram_mean_5',
    'chromagram_mean_6',
    'chromagram_mean_7',
    'chromagram_mean_8',
    'chromagram_mean_9',
    'chromagram_mean_10',
    'chromagram_mean_11',
    'chromagram_std_0',
    'chromagram_std_1',
    'chromagram_std_2',
    'chromagram_std_3',
    'chromagram_std_4',
    'chromagram_std_5',
    'chromagram_std_6',
    'chromagram_std_7',
    'chromagram_std_8',
    'chromagram_std_9',
    'chromagram_std_10',
    'chromagram_std_11',
    'pitch_salience.mean',
    'pitch_salience.std',
    'tonal.chords_hist_0',
    'tonal.chords_hist_1',
    'tonal.chords_hist_2',
    'tonal.chords_hist_3',
    'tonal.chords_hist_4',
    'tonal.chords_hist_5',
    'tonal.chords_hist_6',
    'tonal.chords_hist_7',
    'tonal.chords_hist_8',
    'tonal.chords_hist_9',
    'tonal.chords_hist_10',
    'tonal.chords_hist_11',
    'tonal.chords_hist_12',
    'tonal.chords_hist_13',
    'tonal.chords_hist_14',
    'tonal.chords_hist_15',
    'tonal.chords_hist_16',
    'tonal.chords_hist_17',
    'tonal.chords_hist_18',
    'tonal.chords_hist_19',
    'tonal.chords_hist_20',
    'tonal.chords_hist_21',
    'tonal.chords_hist_22',
    'tonal.chords_hist_23',
    'tonal.key_0',
    'tonal.key_1',
    'tonal.key_2',
    'tonal.key_3',
    'tonal.key_4',
    'tonal.key_5',
    'tonal.key_6',
    'tonal.key_7',
    'tonal.key_8',
    'tonal.key_9',
    'tonal.key_10',
    'tonal.key_11',
    'tonal.scale',
    'tonal.key_strength']

In [ ]:
# replace audio feature names in df
new_cols = dict(zip(df.columns[1:-10], better_feat_names)) # for 200-10 model
# new_cols = dict(zip(df.columns[1:-5], better_feat_names)) # for 200-5 model
df.rename(columns= new_cols, inplace=True)

In [ ]:
# Adapt 10-label dataset to 5 labels

pairs = [['care','harm'],['fairness','cheating'],['loyalty','betrayal'],['authority','subversion'],['purity','degradation']]

# Iterate over each pair
for pair in pairs:
    col1, col2 = pair

    # Create the new combined column
    df[col1] = ((df[col1] == 1) | (df[col2] == 1)).astype(int)

    # Drop the original columns
    df.drop([col2], axis=1, inplace=True)
    # print(f"{df[col1].value_counts()}\n")

In [ ]:
# plot class distributions for each moral value label

# column_names = list(df_plot[mft_list])
# column_names = list(df_plot[mft_list_5])
# column_names = ['Ca', 'H', 'F', 'Ch', 'L', 'B', 'A', 'S', 'P', 'D']
column_names = ['Ca/H', 'F/Ch', 'L/B', 'A/S', 'P/D']
percent_names = ['', '25%', '50%', '75%', '100%']

# Count the occurrences of 0s and 1s in each column
zeros_count = df[mft_list_5].apply(lambda x: (x == 0).sum()).values
ones_count = df[mft_list_5].apply(lambda x: (x == 1).sum()).values

# Convert to percentages
total_counts = df[mft_list_5].count().values
zeros_percent = (zeros_count / total_counts) * 100
ones_percent = (ones_count / total_counts) * 100

# Set up the figure and axis
fig, ax = plt.subplots(figsize=(4, 3))

# Create the stacked bar plot
bar_width = 0.6
x = np.arange(len(column_names))

# Plot the counts of 1s (bottom bars)
p1 = ax.bar(x, ones_percent, bar_width, label='present', color='thistle', zorder=3)

# Plot the counts of 0s (stacked on top)
p2 = ax.bar(x, zeros_percent, bar_width, bottom=ones_percent, label='absent', color='peachpuff', zorder=3)

# Add labels, title, and legend
# ax.set_xlabel('Columns', fontsize=12)
# ax.set_ylabel('Count', fontsize=12)
# ax.set_title('Distribution of 1s and 0s Across 10 classes', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(column_names, fontsize=16)
ax.set_yticks(np.arange(0, 101, 25))
ax.set_yticklabels([], fontsize=16)
# ax.legend(fontsize=16)
plt.grid(axis = 'y', zorder=0)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
# ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.tick_params(axis="y", left=False)

# plt.savefig(f'distribution_5.pdf')

### Model training with RFECV and Optuna

In [ ]:
#@title RFECV function
# code from: https://github.com/marinelliluca/explainable-modeling

def feature_selection(X: pd.DataFrame, y: pd.Series) -> RFECV:
    """Perform feature selection using RFECV with XGBClassifier."""

    rfecv = RFECV(
        estimator=XGBClassifier(),
        step=1,
        cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
        scoring="f1",
        # scoring="f1_weighted",
        min_features_to_select=1,
        n_jobs=-1
    )
    rfecv.fit(X, y)
    
    return rfecv

In [ ]:
#@title Optuna objective
# code based on: https://github.com/marinelliluca/explainable-modeling

def objective(trial: optuna.Trial, data: pd.DataFrame, target: pd.Series) -> float:
    """Objective function for Optuna optimization."""

    # Calculate the pos_weight and its sqrt
    pos_weight = (target == 0).sum() / (target == 1).sum()
    pos_weight = np.round(pos_weight, 1)
    sqrt_pos_weight = np.round(np.sqrt(pos_weight), 1)

    params = {
        "verbosity": 0,
        "objective": "binary:logistic",
        "tree_method": "exact", # for small datasets
        "n_estimators": trial.suggest_int("n_estimators", 100, 300, step=100),
        "max_depth": trial.suggest_categorical("max_depth", [3, 5, 6]),
        "learning_rate": trial.suggest_categorical("learning_rate", [0.01, 0.1, 0.2]),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0, step=0.2, log=False),
        "lambda": trial.suggest_float("lambda", 0.5, 1.5, step=0.5, log=False),
        "alpha": trial.suggest_float("alpha", 0.5, 1.5, step=0.5, log=False),
        # Add the scale_pos_weight
        "scale_pos_weight": trial.suggest_categorical("scale_pos_weight", [1.0, sqrt_pos_weight, pos_weight]),
        # prevent the model from overcorrecting for the minority class
        "max_delta_step": trial.suggest_categorical("max_delta_step", [0.0, 1.0, 5.0])
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    target_pred = cross_val_predict(XGBClassifier(**params), data, target, cv=cv, n_jobs=-1)
    f1_binary = f1_score(target, target_pred)
    # f1_weighted = f1_score(target, target_pred, average="weighted")
    return f1_binary
    # return f1_weighted

In [ ]:
#@title run RFECV and OPTUNA and save optimal features and best models
# based on code from: https://github.com/marinelliluca/explainable-modeling

for mft in mft_list:  # use mft_list_5 for 5 labels
    # Separate features and target
    X = df.iloc[:, 1:-10]  # Features (adjust this based on 10-label or 5-label dataset)
    # X = df.iloc[:, 1:-5]
    y = df[mft]  # Target for the current MFT

    # recursive feature elimination with cross-validation
    rfecv = feature_selection(X, y)
    print(f"Optimal number of features: {rfecv.n_features_}") 
    selected_features = X.columns[rfecv.support_].tolist()
    print(f"Selected features: {selected_features}")

    # Optuna optimization
    study = optuna.create_study(direction="maximize")
    study.optimize(
        lambda trial: objective(trial, X[selected_features], y),
        # gc_after_trial=True,
        n_trials=200
        )

    print("\nNumber of finished trials: ", len(study.trials))
    print("\nBest trial:")
    trial = study.best_trial
    print("\tValue: ", trial.value)
    print("\tParams: ")
    for key, value in trial.params.items():
        print(f"\t\t{key}: {value}")

    fn = f"models_f1_binary/xgboost_params_200-10-FE_{mft}.json"
    print(f"\nSaving best trial to {fn}")
    with open(fn, "w") as f:
        json.dump(study.best_params, f)

    X_selected = df[selected_features + [mft]]
    fn = f"features_f1_binary/selected_features_200-10-FE_{mft}.csv"
    print(f"Saving {fn}")
    X_selected.to_csv(fn)

### Test best models

In [ ]:
# based on code from: https://github.com/marinelliluca/explainable-modeling

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)  # 5-fold cross-validation

for mft in mft_list:  # use mft_list_5 for 5 labels
    # Separate features and target
    X = df.iloc[:, 1:-10]  # Features (adjust this based on 10-label or 5-label dataset)
    # X = df.iloc[:, 1:-5]
    y = df[mft]  # Target for the current MFT

    # LOAD OPTIMAL FEATURES
    fn = f"features_f1_binary/selected_features_200-10-FE_{mft}.csv"
    features_df = pd.read_csv(fn)
    best_features = list(features_df.iloc[:, 1:-1])
    X = df[best_features]
    y = df[mft]
    
    # HYPERPARAMETER TUNING
    # import the best parameters from an optuna study with 200 trials
    fn = f"models_f1_binary/xgboost_params_200-10-FE_{mft}.json"
    with open(fn) as json_file:
      best_params = json.load(json_file)

    # 5-FOLD CV
    base_params = {
        "verbosity": 0,
        "objective": "binary:logistic",
        # use exact for small dataset
        "tree_method": "exact"
        }
    best_params.update(base_params)
    model = XGBClassifier(**best_params)
    # y_pred = cross_val_predict(model, X, y, cv=cv)
    results = cross_val_score(model, X, y, cv=cv, scoring="f1")
    # results = cross_val_score(model, X, y, cv=cv, scoring="f1_weighted")

    # MEAN F1 SCORES
    f1_mean = np.mean(results)
    f1_std = np.std(results)
    # Print the results
    print(f'MFT: {mft}, Mean F1: {f1_mean:.2f}, Std: {f1_std:.2f}')
    print('\n')

### SAGE analysis

In [ ]:
# code from: https://github.com/marinelliluca/explainable-modeling

# load optimal features from EN200 experiments
mft = "care"
fn = f"features_f1_binary/selected_features_200-5-FE_{mft}.csv"
features_df = pd.read_csv(fn)
best_features = list(features_df.iloc[:, 1:-1])
X = df[best_features]
y = df[mft]

# import the best parameters from an optuna study with 200 trials
fn = f"models_f1_binary/xgboost_params_200-5-FE_{mft}.json"
with open(fn) as json_file:
  best_params = json.load(json_file)

# 5-FOLD CV
base_params = {
    "verbosity": 0,
    "objective": "binary:logistic",
    # use exact for small dataset
    "tree_method": "exact"
    }
best_params.update(base_params)

# Train the model on the whole dataset
model = XGBClassifier(**best_params).fit(X, y)

# Set up an imputer to handle missing features
imputer = sage.MarginalImputer(model, X.values)

# Set up an estimator
estimator = sage.PermutationEstimator(imputer, loss='cross entropy', random_state=42)

# Calculate SAGE values
sage_values = estimator(X.values, y)

sage_values_df = pd.DataFrame(columns=['feature name','SAGE value'])
sage_values_df['feature name'] = X.columns
sage_values_df['SAGE value'] = sage_values.values
sage_values_df.sort_values(by=['SAGE value'], ascending=False, inplace=True)

# give SAGE order to SHAP beeswarm plot
sage_ordered_features = sage_values_df['feature name'].tolist()

col2num = {col: i for i, col in enumerate(X.columns)}

In [ ]:
# plot SAGE values

import seaborn as sns

# Create figure
plt.figure()
# fig, ax = plt.subplots()
sns.barplot(x="SAGE value", y="feature name", data=sage_values_df,
            order=sage_ordered_features[:9], color="darkmagenta")
#pl.title('Feature importance')
plt.ylabel('')
# plt.xticks(fontsize=16)
# plt.yticks(fontsize=16)
plt.tick_params(labelsize=16)
plt.tight_layout()
plt.xlabel('SAGE value', fontsize=16)
sns.despine(top=True, right=True, left=False, bottom=False)

# plt.savefig(f'sage_{mft}.pdf')
# plt.savefig(f'sage_{mft}_harm.pdf')
# plt.show()
# plt.close()


### SHAP analysis

In [ ]:
# code from: https://github.com/marinelliluca/explainable-modeling

# load optimal features from EN200 experiments
mft = "care"
fn = f"features_f1_binary/selected_features_200-10-FE_{mft}.csv"
features_df = pd.read_csv(fn)
best_features = list(features_df.iloc[:, 1:-1])
X = df[best_features]
y = df[mft]

# import the best parameters from an optuna study with 200 trials
fn = f"models_f1_binary/xgboost_params_200-10-FE_{mft}.json"
with open(fn) as json_file:
  best_params = json.load(json_file)

# 5-FOLD CV
base_params = {
    "verbosity": 0,
    "objective": "binary:logistic",
    # use exact for small dataset
    "tree_method": "exact"
    }
best_params.update(base_params)

# Train the model on the whole dataset
model = XGBClassifier(**best_params).fit(X, y)

# Set up colours for Beeswarm plot
colors = ["darkmagenta","orange"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)

explainer = shap.Explainer(model, feature_names=X.columns)
shap_values = explainer(X)

# Create figure
# plt.figure(figsize=(8,18))
# fig, ax = plt.subplots()
plt.figure()

# summarize the effects of all the features
ax = shap.plots.beeswarm(
    shap_values,
    # order=shap_values.abs.max(0),
    # order=[col2num[col] for col in sage_ordered_features],
    # show= True,
    max_display=22,
    color_bar=False,
    alpha = 1,
    color = cmap,
    group_remaining_features = False
)

plt.tick_params(labelsize=17)
plt.tight_layout()
plt.xlabel('SHAP value', fontsize=17)

# plt.savefig(f'shap_{mft}.pdf')
# plt.savefig(f'shap_{mft}_harm.pdf')
# plt.show()

### constant classifiers

In [ ]:
# we loop through each MFT column
for mft in mft_list:

    # Separate features (audio columns) and target (current MFT column)
    X = df.iloc[:, 1:-10]  # audio feature columns
    y = df[mft]  # current MFT column as the target

    model = DummyClassifier(strategy='most_frequent')
    # model = DummyClassifier(strategy='constant', constant=1)
    model.fit(X, y)
    y_pred = model.predict(X)

    # CALCULATE F1 SCORE
    f1_weighted = f1_score(y, y_pred, average='weighted')
    # f1_binary = f1_score(y, y_pred, average='binary')
   
    # Print the results
    # print(f'MFT: {mft}, F1 Binary:{f1_binary:.2f}')
    print(f'MFT: {mft}, F1 Weighted:{f1_weighted:.2f}')
    print('\n')